# Eksperimen 2: Pelatihan Model Hibrid RT-DETR + ConvNeXt V2 + CBAM

**Judul Skripsi:**  
IMPLEMENTASI HIBRID RT-DETR DAN CONVNEXT V2 TERINTEGRASI CBAM UNTUK KLASIFIKASI DAN KUANTIFIKASI KUMPULAN BIJI FAMILI FABACEAE

**Mahasiswa:** Refael Juari Siagian (NIM: 231402055)  
**Hardware:** NVIDIA GeForce RTX 4070 (12 GB VRAM)  
**Dataset:** Biji Famili Fabaceae (*Albizia lophantha*, *Clitoria ternatea*)

---  
## 1. Verifikasi Lingkungan & Hardware GPU

In [ ]:
import sys
from pathlib import Path

# Pastikan ultralytics_repo terhubung langsung ke sys.path
repo_path = str(Path('ultralytics_repo').resolve())
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

import torch
import ultralytics

print("=" * 65)
print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Tersedia     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Nama GPU          : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Total VRAM GPU    : {vram_gb:.2f} GB")
print("=" * 65)
ultralytics.checks()

---  
## 2. Verifikasi File Konfigurasi Dataset (`dataset/data.yaml`)
Cell ini memeriksa kelengkapan file citra di folder `train/`, `valid/`, dan `test/` serta membaca nama-nama kelas spesies biji Fabaceae.

In [ ]:
import os
try:
    import yaml
except ImportError:
    import subprocess, sys
    print("Menginstal pyyaml otomatis...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyyaml"])
    import yaml

data_yaml_path = "dataset/data.yaml"
with open(data_yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)

classes = data_cfg.get("names", [])
print("=" * 65)
print(f"Path Dataset      : {data_cfg.get('path')}")
print(f"Jumlah Kelas (nc) : {len(classes)}")
print(f"Daftar Spesies    : {classes}")
print("-" * 65)

valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".heic", ".heif")
for split in ["train", "valid", "test"]:
    img_dir = os.path.join("dataset", split, "images")
    lbl_dir = os.path.join("dataset", split, "labels")
    if os.path.exists(img_dir):
        n_imgs = len([f for f in os.listdir(img_dir) if f.lower().endswith(valid_exts)])
        n_lbls = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt') and not f.startswith('.')]) if os.path.exists(lbl_dir) else 0
        print(f"Split {split:<6} : {n_imgs:>4} citra  |  {n_lbls:>4} file label (.txt)")
    else:
        print(f"Split {split:<6} : Folder {img_dir} belum ditemukan.")
print("=" * 65)

---  
## 3. Inisialisasi Model Hibrid (RT-DETR + ConvNeXt V2 + CBAM)
Memuat arsitektur model dari file konfigurasi `configs/rtdetr_convnextv2_cbam.yaml`.

In [ ]:
from ultralytics import RTDETR

model_yaml = "configs/rtdetr_convnextv2_cbam.yaml"
print(f"Menginisialisasi arsitektur dari: {model_yaml}")
model = RTDETR(model_yaml)
print("Model hibrid berhasil disusun!")

---  
## 4. Pelatihan Model (*Training Pipeline*) di GPU NVIDIA RTX 4070

Pengaturan hiperparameter (sesuai Bab 3 / Dokumen Metodologi Skripsi):
- **`imgsz=640`**: Resolusi standar dengan letterboxing otomatis untuk menjaga rasio morfologi asli biji.
- **`epochs=50`**: Jumlah putaran iterasi pelatihan (bisa ditambah ke 100 untuk eksperimen final).
- **`batch=8`**: Ukuran batch optimal untuk VRAM 12 GB GPU RTX 4070.
- **`optimizer='AdamW'`**: Pengoptimal dengan pembobotan *weight decay* yang stabil untuk model transformer.
- **`device=0`**: Menjalankan pelatihan pada GPU NVIDIA GeForce RTX 4070.

In [ ]:
# Jalankan proses pelatihan
results = model.train(
    data="dataset/data.yaml",
    epochs=50,             # Jumlah epoch pelatihan
    imgsz=640,             # Ukuran citra masukan
    batch=8,               # Ukuran batch per iterasi
    workers=4,             # Jumlah worker CPU untuk memuat data
    device=0,              # GPU RTX 4070
    optimizer="AdamW",      # Optimizer AdamW sesuai metodologi
    project="runs/train",   # Direktori utama penyimpanan hasil
    name="rtdetr_convnextv2_cbam_fabaceae", # Nama folder eksperimen
    save=True,             # Simpan bobot terbaik (best.pt)
    plots=True             # Buat grafik otomatis (loss, mAP, confusion matrix)
)

print("Pelatihan selesai! Bobot terbaik tersimpan di runs/train/rtdetr_convnextv2_cbam_fabaceae/weights/best.pt")

---  
## 5. Evaluasi Kinerja pada Data Uji (*Testing Set*)
Cell ini menguji model terbaik (**`best.pt`**) pada data uji murni (*test split*) untuk mengukur metrik standar Computer Vision:
- **mAP@50** dan **mAP@50-95** (Akurasi Lokalisasi & Deteksi Kotak)
- **Precision** dan **Recall**
- **F1-Score**

In [ ]:
best_weights = "runs/train/rtdetr_convnextv2_cbam_fabaceae/weights/best.pt"

# Muat model dengan bobot terbaik hasil training
trained_model = RTDETR(best_weights)

# Evaluasi pada split data uji (test set)
metrics = trained_model.val(data="dataset/data.yaml", split="test")

print("=" * 65)
print("HASIL EVALUASI METRIK PADA DATA UJI (TEST SET):")
print(f"mAP@50       : {metrics.box.map50:.4f}")
print(f"mAP@50-95    : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")
f1 = 2 * (metrics.box.mp * metrics.box.mr) / (metrics.box.mp + metrics.box.mr + 1e-16)
print(f"F1-Score     : {f1:.4f}")
print("=" * 65)

---  
## 6. Kuantifikasi Biji & Visualisasi Prediksi Deteksi
Sesuai tujuan skripsi untuk melakukan **klasifikasi sekaligus kuantifikasi (penghitungan jumlah total butir biji)** pada kumpulan biji Fabaceae.

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt

# Ambil citra dari data uji
test_images = glob.glob("dataset/test/images/*.jpg") + glob.glob("dataset/test/images/*.png")

if test_images:
    sample_img_path = test_images[0]
    pred_results = trained_model.predict(sample_img_path, conf=0.25, device=0)
    
    # Hitung jumlah biji terdeteksi per kelas
    detected_boxes = pred_results[0].boxes
    total_count = len(detected_boxes)
    
    print("=" * 65)
    print(f"Citra Uji                  : {os.path.basename(sample_img_path)}")
    print(f"Total Butir Biji Terdeteksi: {total_count} butir")
    print("=" * 65)
    
    # Tampilkan gambar hasil prediksi dengan bounding box & label spesies
    res_plot = pred_results[0].plot()
    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(res_plot, cv2.COLOR_BGR2RGB))
    plt.title(f"Hasil Deteksi & Kuantifikasi: {total_count} Biji Terdeteksi")
    plt.axis("off")
    plt.show()
else:
    print("Belum ada citra di folder dataset/test/images/ untuk divisualisasikan.")